# Filter Object JSONs

In [ ]:
# REMOVE DINO

import json
from os import path

from Museum import Museum
from params.collections import MUSEUMS

def filter_objects(museum_info, keep=[]):
  if type(keep) is not list:
    keep = [keep]

  Museum.prep_dirs(museum_info)
  museum_data = Museum.read_data()

  qids = sorted(list(museum_data.keys()))

  for cnt,qid in enumerate(qids):
    object_path = path.join(Museum.DIRS["objects"], f"{qid}.json")

    if (not path.isfile(object_path)):
      continue

    with open(object_path, "r", encoding="utf-8") as ifp:
      object_data = json.load(ifp)

    object_data_filtered = { qid: { k:v for k,v in object_data[qid].items() if k in keep } }

    with open(object_path, "w", encoding="utf-8") as of:
      json.dump(object_data_filtered, of, sort_keys=True, separators=(",",":"), ensure_ascii=False)


In [ ]:
MUSEUMS = { k:v for k,v in MUSEUMS.items() if k != "file" }
for name,info in MUSEUMS.items():
  print("objects:", name)
  filter_objects(info, "owlv2")

# Test Owlv2

In [ ]:
# TEST OWLv2

from PIL import Image as PImage, ImageDraw as PImageDraw

from models.Owlv2 import Owlv2
from params.detect import Owlv2Objects

mo = Owlv2()

qid = "5400323"
qid = "Q10301958"
# qid = "Q19859152"
# qid = "Q677682"
# qid = "Q124157604"
# qid = "MU16352"
# qid = "Q28801524"

img = PImage.open(f"../../imgs/arts/900/{qid}.jpg")
dimg = img.copy()
draw = PImageDraw.Draw(dimg)
iw,ih = img.size

objs = []
for il,t in zip(Owlv2Objects.OBJECT_LABELS_IN, Owlv2Objects.OBJECT_THOLDS):
  # objs += mo.run_object_detection(img, il, t)
  objs += mo.iou_objects(img, il, t, iou_per_label=False)

for o in objs:
  x0,y0,x1,y1 = o["box"]
  draw.rectangle((x0*iw, y0*ih, x1*iw, y1*ih), outline="green")
  draw.text((x0*iw+2, y0*ih), o["label"], stroke_fill="black", fill="white", stroke_width=2)

dimg

# Test Dino

In [ ]:
# TEST DINO

from os import listdir
from random import sample

from PIL import Image as PImage, ImageDraw as PImageDraw

from models.Dino import Dino
from params.detect import DinoObjects

md = Dino(all_labels=DinoObjects.OBJECT_LABELS_OUT)

In [ ]:
# qid = "5400323"
# qid = "Q10301958"
# qid = "Q19859152"
# qid = "Q677682"
# qid = "Q124157604"
# qid = "MU16352"
# qid = "Q28801524"
# qid = "MU16471"
qid = "Q61885913"
# qid = "Q130335947"

# qid = "MU20235"
# qid = "MU18269"
# qid = "MU17444"
# qid = "MU17663"
# qid = "MU19845"
# qid = "MU17514"

img = PImage.open(f"../../imgs/arts/900/{qid}.jpg")
# iw,ih = img.size
# sf = 1280/max(iw,ih)
# nw,nh = int(iw*sf), int(ih*sf)
# img = img.resize((nw,nh))
dimg = img.copy()
draw = PImageDraw.Draw(dimg)
iw,ih = img.size

objs = []
for il,t,ol in zip(DinoObjects.OBJECT_LABELS_IN, DinoObjects.OBJECT_THOLDS, DinoObjects.OBJECT_LABELS_OUT):
  # objs += md.run_object_detection(img, il, t, ol)
  objs += md.iou_objects(img, il, t, ol, iou_per_label=True)

for o in objs:
  x0,y0,x1,y1 = o["box"]
  draw.rectangle((x0*iw, y0*ih, x1*iw, y1*ih), outline="green")
  draw.text((x0*iw+2, y0*ih), o["label"], stroke_fill="black", fill="white", stroke_width=2)

dimg

In [ ]:
# EXPORT IMAGES

img_dir = "../../imgs/arts/900"
all_img_paths = sorted([f"{img_dir}/{f}" for f in listdir(img_dir) if f.endswith(".jpg")])
img_paths = sample(all_img_paths, 128)

for cnt,f in enumerate(img_paths):
  img = PImage.open(f)
  dimg = img.copy()
  draw = PImageDraw.Draw(dimg)
  iw,ih = img.size
  objs = []
  for il,t,ol in zip(DinoObjects.OBJECT_LABELS_IN, DinoObjects.OBJECT_THOLDS, DinoObjects.OBJECT_LABELS_OUT):
    objs += md.run_object_detection(img, il, t, ol)

  if cnt%10 == 0:
    print(cnt, "/", len(img_paths))
    for o in objs:
      x0,y0,x1,y1 = o["box"]
      draw.rectangle((x0*iw, y0*ih, x1*iw, y1*ih), outline="green")
      draw.text((x0*iw+2, y0*ih), o["label"], stroke_fill="black", fill="white", stroke_width=2)
    display(dimg)

# Test Florence2

In [ ]:
# TEST Florence2

from PIL import Image as PImage, ImageDraw as PImageDraw

from models.Florence2 import Florence2
from params.detect import DinoObjects

mo = Florence2()

In [ ]:
qid = "5400323"
qid = "Q10301958"
qid = "Q19859152"
qid = "Q677682"
qid = "Q124157604"
# qid = "MU16352"
# qid = "Q28801524"

img = PImage.open(f"../../imgs/arts/900/{qid}.jpg")
dimg = img.copy()
draw = PImageDraw.Draw(dimg)
iw,ih = img.size

objs = []
for il in list(DinoObjects.OBJECT_LABELS_IN2OUT.keys()):
  objs += mo.run_object_detection(img, il)

for o in objs:
  x0,y0,x1,y1 = o["box"]
  draw.rectangle((x0*iw, y0*ih, x1*iw, y1*ih), outline="green")
  draw.text((x0*iw+2, y0*ih), o["label"], stroke_fill="black", fill="white", stroke_width=2)

dimg